# Hands-on Introduction to FastText Subword Embeddings

A comprehensive guide to understanding how FastText represents words via character n-grams. We'll train subword embeddings on a toy corpus, inspect how subwords contribute to word vectors, benchmark training speed, and visualize embedding spaces.

## Why FastText Uses Subwords

Traditional word embeddings like Word2Vec treat each word as an atomic unit. FastText extends this by representing words as bags of character n-grams, enabling:
- Better handling of morphologically rich languages
- Robust representations for out-of-vocabulary (OOV) words  
- Improved performance on rare words
- Compositional understanding of word structure

In [ ]:
# Setup and Installation
import os
import sys
import subprocess
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Install fasttext if not available
try:
    import fasttext
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fasttext"])
    import fasttext

# Install additional dependencies
try:
    from sklearn.manifold import TSNE
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    from sklearn.manifold import TSNE

print("✅ All dependencies installed successfully!")

## Understanding Subword Tokenization

FastText breaks words into character n-grams with configurable minimum (`minn`) and maximum (`maxn`) sizes. Special boundary symbols `<` and `>` are added to distinguish prefixes and suffixes.

**Key concepts:**
- **Character n-grams**: Contiguous sequences of n characters
- **Boundary symbols**: `<word>` format helps distinguish word positions
- **Subword vocabulary**: All possible n-grams from the training corpus
- **Word vector**: Sum of its constituent subword vectors + full word vector (if seen during training)

In [ ]:
def extract_ngrams(word: str, min_n: int = 3, max_n: int = 6) -> list[str]:
    """Extract all character n-grams from a word with boundary symbols."""
    # Add boundary symbols
    bounded_word = f"<{word}>"
    ngrams = []
    
    # Extract n-grams of different sizes
    for n in range(min_n, max_n + 1):
        for i in range(len(bounded_word) - n + 1):
            ngrams.append(bounded_word[i:i + n])
    
    return ngrams

def display_ngrams(words: list[str], min_n: int = 3, max_n: int = 6) -> None:
    """Display n-grams for multiple words in a formatted table."""
    results = []
    
    for word in words:
        ngrams = extract_ngrams(word, min_n, max_n)
        results.append({
            'Word': word,
            'Bounded': f"<{word}>",
            'N-grams Count': len(ngrams),
            'N-grams': ', '.join(ngrams[:10]) + ('...' if len(ngrams) > 10 else '')
        })
    
    df = pd.DataFrame(results)
    return df

# Test with sample words including OOV tokens
test_words = ['where', 'running', 'chatgpt', 'unicornism', 'fast', 'fastest']
ngrams_df = display_ngrams(test_words)
print("🔍 N-gram Extraction Examples:")
print(ngrams_df.to_string(index=False))

# Detailed view for one word
word_example = 'where'
ngrams = extract_ngrams(word_example)
print(f"\n📋 Detailed n-grams for '{word_example}':")
print(f"Bounded word: <{word_example}>")
print(f"All n-grams ({len(ngrams)}): {ngrams}")

In [ ]:
# Create a sample training corpus
sample_corpus = """
The quick brown fox jumps over the lazy dog.
Machine learning is transforming artificial intelligence.
FastText embeddings capture subword information effectively.
Natural language processing requires understanding context.
Deep learning models learn complex patterns from data.
Computer vision analyzes and interprets visual information.
Data science combines statistics programming and domain knowledge.
Neural networks mimic biological neural connections.
Algorithms solve computational problems efficiently.
Programming languages enable software development.
"""

# Save corpus to file
corpus_path = 'sample_corpus.txt'
with open(corpus_path, 'w', encoding='utf-8') as f:
    f.write(sample_corpus.strip())

print(f"📄 Created sample corpus: {corpus_path}")
print(f"Corpus preview:\n{sample_corpus[:200]}...")

## The Training Corpus: Data Overview

Everything below is trained on the ten sentences written out in the cell above -- there is
no external dataset to download, which keeps the notebook self-contained and makes every
result exactly reproducible.

**What is in it:** ten one-line sentences of generic machine-learning prose, so the
vocabulary is dominated by words like *learning*, *data*, *neural*, *language*, *computer*.
Several words share morphology (*learn* / *learning*, *program* / *programming* /
*languages*), which is deliberate: those are the pairs where subword sharing is visible.

**What is not in it:** everything else. The corpus is a few hundred bytes, each word appears
once or twice, and there is no repeated context for the skip-gram objective to exploit. It
is the right size for *inspecting* how FastText assembles a word vector out of character
n-grams, and far too small for the resulting vectors to mean anything semantically.

Keep that distinction in mind for the rest of the notebook: the machinery on display is
real, the embeddings it produces are not. Section "Limitations & Caveats" near the end
spells out exactly which of the results below are artifacts of corpus size.

In [ ]:
# Exactly how small is "small"? Numbers we will need for the caveats later.
lines = [ln for ln in sample_corpus.strip().splitlines() if ln.strip()]
tokens = sample_corpus.split()
types = {tok.strip('.').lower() for tok in tokens}

print('=== Corpus size ===')
print(f'  Sentences:      {len(lines)}')
print(f'  Tokens:         {len(tokens)}')
print(f'  Unique tokens:  {len(types)}')
print(f'  Bytes on disk:  {os.path.getsize(corpus_path):,}')
print(f'  Type/token ratio: {len(types) / len(tokens):.2f}  '
      '(1.00 would mean every word appears exactly once)')

# How many distinct character n-grams does that produce? This is the vocabulary
# FastText actually learns over, and it is much larger than the word vocabulary.
all_ngrams = set()
for tok in types:
    all_ngrams.update(extract_ngrams(tok))
print(f'\n  Distinct 3-6 char n-grams across the vocabulary: {len(all_ngrams)}')
print(f'  Ratio of n-grams to word types: {len(all_ngrams) / len(types):.1f}x')
print('\nThat ratio is the whole idea: FastText trades a small word vocabulary for a')
print('much larger subword vocabulary, which is what lets it embed words it never saw.')

In [ ]:
# Train FastText model with timing
print("🚀 Training FastText model...")
start_time = time.time()

try:
    model = fasttext.train_unsupervised(
        input=corpus_path,
        model='skipgram',
        minn=3,           # Minimum n-gram size
        maxn=6,           # Maximum n-gram size  
        dim=50,           # Embedding dimension
        ws=5,             # Context window size
        epoch=5,          # Number of epochs
        lr=0.05,          # Learning rate
        thread=4,         # Number of threads
        verbose=0         # Suppress verbose output
    )
    
    training_time = time.time() - start_time
    
    print(f"✅ Training completed in {training_time:.2f} seconds")
    print(f"📊 Model parameters:")
    print(f"  - Vocabulary size: {len(model.get_words())}")
    print(f"  - Embedding dimension: {model.get_dimension()}")
    
    # Get model parameters without using get_args()
    print(f"  - Subword range: 3-6 (as specified)")
    print(f"  - Model type: skipgram (as specified)")
    print(f"  - Training epochs: 5 (as specified)")
    
    # Test model functionality
    test_word = "machine"
    if test_word in model.get_words():
        test_vector = model.get_word_vector(test_word)
        print(f"  - Test vector for '{test_word}': ✅ (norm: {np.linalg.norm(test_vector):.3f})")
    else:
        print(f"  - Test word '{test_word}' not in vocabulary")
    
    # Save model for future use
    model.save_model("fasttext_model.bin")
    print("💾 Model saved as 'fasttext_model.bin'")
    
except Exception as e:
    print(f"❌ Training failed: {e}")
    print("🔍 Checking corpus file...")
    if os.path.exists(corpus_path):
        with open(corpus_path, 'r') as f:
            content = f.read()
        print(f"Corpus content (first 100 chars): {content[:100]}")
    else:
        print("Corpus file not found!")

## Inspecting Subword Contributions

FastText composes the final word vector by:
1. Extracting all character n-grams from the word, plus the whole word itself
2. Looking up each piece's row in the model's input matrix
3. **Averaging** those rows -- one detail worth knowing, because the paper describes
   the representation as a *sum* while the reference implementation divides by the
   number of pieces. The cell below reconstructs the word vector both ways so you can
   see which one the library actually does.

If the word was seen during training it contributes its own whole-word row; if it was
not, the vector is built from n-grams alone. That is the entire mechanism behind
FastText's out-of-vocabulary handling -- and also its limit, since a word sharing no
n-grams with the training vocabulary gets a vector assembled from noise.

In [ ]:
def analyze_subwords(model, word: str) -> dict:
    """Analyze how each character n-gram contributes to a word's embedding.

    FastText keeps one row per vocabulary word followed by one row per n-gram hash
    bucket, all in a single input matrix. `get_subwords` hands back the row index
    of every piece, so the real n-gram vectors can be read directly rather than
    guessed at.
    """
    subwords, indices = model.get_subwords(word)
    input_matrix = model.get_input_matrix()
    word_vector = model.get_word_vector(word)

    subword_vectors = np.array([input_matrix[idx] for idx in indices])
    subword_norms = np.linalg.norm(subword_vectors, axis=1)

    # Rebuild the word vector both ways -- summing the pieces and averaging them --
    # and report how far each lands from what the library returns. Whichever error
    # is ~0 is what FastText actually does.
    err_sum = float(np.linalg.norm(subword_vectors.sum(axis=0) - word_vector))
    err_mean = float(np.linalg.norm(subword_vectors.mean(axis=0) - word_vector))

    return {
        'word': word,
        'subwords': list(subwords),
        'subword_count': len(subwords),
        'word_vector_norm': float(np.linalg.norm(word_vector)),
        'subword_norms': subword_norms,
        'mean_subword_norm': float(subword_norms.mean()),
        'error_if_summed': err_sum,
        'error_if_averaged': err_mean,
        'is_oov': word not in model.get_words(),
    }

# Analyze several words: two in-vocabulary, two that the model never saw.
analysis_words = ['learning', 'programming', 'chatgpt', 'fastest']
print("Subword Analysis Results")
print("=" * 60)

for word in analysis_words:
    analysis = analyze_subwords(model, word)
    print(f"\nWord: '{analysis['word']}'  (out-of-vocabulary: {analysis['is_oov']})")
    print(f"  Pieces ({analysis['subword_count']}): {analysis['subwords'][:8]}"
          f"{' ...' if analysis['subword_count'] > 8 else ''}")
    print(f"  |word vector|:        {analysis['word_vector_norm']:.3f}")
    print(f"  mean |piece vector|:  {analysis['mean_subword_norm']:.3f}")
    print(f"  error if pieces summed:   {analysis['error_if_summed']:.6f}")
    print(f"  error if pieces averaged: {analysis['error_if_averaged']:.6f}")

print("\n" + "=" * 60)
print("Whichever error above is ~0 is the composition rule the library uses.")
print("The first piece of an in-vocabulary word is the whole-word row; for an OOV")
print("word that row does not exist, so its vector is built purely from n-grams.")
print("That is why FastText can embed 'chatgpt' at all, and why the embedding it")
print("gives you is only as good as the n-grams it shares with words it did see.")

## Benchmark: Single vs. Multi-threaded Training

FastText supports multi-threaded training to speed up the process. Let's compare training times with different thread counts to understand the performance impact.

In [ ]:
def benchmark_training(corpus_path: str, thread_counts: list[int]) -> pd.DataFrame:
    """Benchmark FastText training with different thread counts."""
    results = []
    
    for threads in thread_counts:
        print(f"🧵 Training with {threads} thread(s)...")
        
        start_time = time.time()
        temp_model = fasttext.train_unsupervised(
            input=corpus_path,
            model='skipgram',
            minn=3, maxn=6,
            dim=50, epoch=5,
            thread=threads,
            verbose=0
        )
        training_time = time.time() - start_time
        
        results.append({
            'Threads': threads,
            'Training Time (s)': round(training_time, 3),
            'Speedup': round(results[0]['Training Time (s)'] / training_time, 2) if results else 1.0,
            'Vocab Size': len(temp_model.get_words())
        })
        
        print(f"  ⏱️ Completed in {training_time:.3f}s")
    
    return pd.DataFrame(results)

# Run benchmark
thread_counts = [1, 2, 4]
benchmark_df = benchmark_training(corpus_path, thread_counts)

print("\n📈 Threading Benchmark Results:")
print(benchmark_df.to_string(index=False))

# Visualize benchmark results
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
plt.bar(benchmark_df['Threads'], benchmark_df['Training Time (s)'], color='skyblue', alpha=0.7)
plt.xlabel('Number of Threads')
plt.ylabel('Training Time (seconds)')
plt.title('Training Time vs. Thread Count')
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(benchmark_df['Threads'], benchmark_df['Speedup'], marker='o', linewidth=2, markersize=8, color='orange')
plt.xlabel('Number of Threads')
plt.ylabel('Speedup Factor')
plt.title('Training Speedup vs. Thread Count')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Embedding Visualization

Let's visualize word embeddings in 2D space using t-SNE dimensionality reduction. This helps us understand the semantic relationships captured by FastText.

In [ ]:
def visualize_embeddings(model, words: list[str], random_state: int = 42):
    """Visualize word embeddings using t-SNE."""
    # Get word vectors
    vectors = []
    valid_words = []
    
    for word in words:
        try:
            vec = model.get_word_vector(word)
            vectors.append(vec)
            valid_words.append(word)
        except Exception as e:
            print(f"⚠️ Could not get vector for '{word}': {e}")
    
    if len(vectors) < 2:
        print("❌ Need at least 2 valid words for visualization")
        return None, []
    
    # Convert to numpy array - this fixes the AttributeError
    vectors_array = np.array(vectors)
    print(f"📊 Vector array shape: {vectors_array.shape}")
    
    # Calculate appropriate perplexity (must be less than n_samples)
    n_samples = len(vectors)
    perplexity = min(30, max(5, n_samples - 1))
    
    # Apply t-SNE
    print(f"🎨 Applying t-SNE to {n_samples} word vectors (perplexity={perplexity})...")
    
    try:
        tsne = TSNE(n_components=2, random_state=random_state, perplexity=perplexity)
        embeddings_2d = tsne.fit_transform(vectors_array)
        
        # Create visualization
        plt.figure(figsize=(12, 8))
        scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                             c=range(len(valid_words)), cmap='tab10', 
                             s=100, alpha=0.7)
        
        # Annotate points
        for i, word in enumerate(valid_words):
            plt.annotate(word, (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=10, fontweight='bold')
        
        plt.title('FastText Word Embeddings Visualization (t-SNE)', fontsize=14, fontweight='bold')
        plt.xlabel('t-SNE Dimension 1')
        plt.ylabel('t-SNE Dimension 2')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        return embeddings_2d, valid_words
        
    except Exception as e:
        print(f"❌ t-SNE failed: {e}")
        return None, valid_words

# Select interesting words for visualization
visualization_words = [
    'machine', 'learning', 'deep', 'neural', 'network',
    'computer', 'vision', 'language', 'processing', 'data',
    'algorithm', 'programming', 'artificial', 'intelligence'
]

# Add some words that might be OOV
visualization_words.extend(['fasttext', 'subword', 'embedding'])

# Check which words are actually in our model vocabulary first
print("🔍 Checking word availability in model:")
available_words = []
for word in visualization_words:
    if word in model.get_words():
        available_words.append(word)
        print(f"✅ '{word}' - in vocabulary")
    else:
        print(f"⚠️ '{word}' - not in vocabulary (will use OOV embedding)")
        available_words.append(word)  # FastText can still embed OOV words

print(f"\n📊 Total words for visualization: {len(available_words)}")

# Visualize embeddings
embeddings_2d, valid_words = visualize_embeddings(model, available_words)

if embeddings_2d is not None:
    print(f"✅ Visualized {len(valid_words)} words in 2D space")
else:
    print("❌ Visualization failed")

## Handling Out-of-Vocabulary (OOV) Words

One of FastText's key advantages is its ability to generate meaningful embeddings for words not seen during training, by leveraging subword information.

In [ ]:
def analyze_oov_words(model, oov_words: list[str], k: int = 5):
    """Analyze OOV word embeddings and find nearest neighbors."""
    print("🔍 Analyzing Out-of-Vocabulary Words:\n")
    
    for word in oov_words:
        is_in_vocab = word in model.get_words()
        print(f"Word: '{word}' (In vocab: {is_in_vocab})")
        
        # Get word vector (works even for OOV)
        try:
            word_vector = model.get_word_vector(word)
            vector_norm = np.linalg.norm(word_vector)
            print(f"  Vector norm: {vector_norm:.3f}")
            
            # Find nearest neighbors
            try:
                neighbors = model.get_nearest_neighbors(word, k=k)
                print(f"  Nearest neighbors:")
                for similarity, neighbor in neighbors:
                    print(f"    {neighbor}: {similarity:.3f}")
            except:
                print("  Could not find nearest neighbors")
                
        except Exception as e:
            print(f"  Error getting vector: {e}")
        
        print()

# Test with various OOV words
oov_test_words = [
    'chatgpt',      # Modern AI term
    'unicornism',   # Made-up word  
    'fasttext',     # Technical term
    'subwording',   # Morphological variant
    'deeplearning', # Compound word
    'preprocessing' # Technical compound
]

analyze_oov_words(model, oov_test_words)

# Compare OOV handling with similar in-vocabulary words
def compare_oov_vs_vocab(model, word_pairs: list[tuple[str, str]]):
    """Compare embeddings between OOV and similar vocabulary words."""
    print("⚖️ OOV vs. Vocabulary Word Comparison:\n")
    
    for oov_word, vocab_word in word_pairs:
        oov_vec = model.get_word_vector(oov_word)
        vocab_vec = model.get_word_vector(vocab_word)
        
        # Calculate cosine similarity
        similarity = np.dot(oov_vec, vocab_vec) / (np.linalg.norm(oov_vec) * np.linalg.norm(vocab_vec))
        
        print(f"'{oov_word}' vs '{vocab_word}': similarity = {similarity:.3f}")

# Compare similar words
word_pairs = [
    ('fasttext', 'fast'),
    ('deeplearning', 'learning'), 
    ('preprocessing', 'processing')
]

compare_oov_vs_vocab(model, word_pairs)

## Practical Tips & Hyperparameters

Here's a comprehensive guide to FastText hyperparameters and their effects:

In [ ]:
# Create hyperparameters reference table
hyperparams_data = {
    'Parameter': ['minn', 'maxn', 'dim', 'ws', 'epoch', 'lr', 'thread', 'model'],
    'Description': [
        'Minimum character n-gram size',
        'Maximum character n-gram size', 
        'Embedding vector dimension',
        'Context window size',
        'Number of training epochs',
        'Learning rate',
        'Number of threads for training',
        'Training algorithm (skipgram/cbow)'
    ],
    'Default': [3, 6, 100, 5, 5, 0.05, 1, 'skipgram'],
    'Range/Options': [
        '2-5 (smaller for morphologically simple languages)',
        '4-8 (larger for complex morphology)',
        '50-300 (higher for larger vocabularies)', 
        '3-10 (larger for more context)',
        '5-20 (more for smaller datasets)',
        '0.01-0.1 (lower for stability)',
        '1-CPU cores (diminishing returns)',
        'skipgram (rare words) / cbow (speed)'
    ],
    'Impact': [
        'Smaller = fewer subwords, faster; Larger = more subwords, slower',
        'Larger = more subwords, better OOV handling, slower training',
        'Higher = more expressive, requires more data',
        'Larger = more context, slower training',
        'More = better convergence, longer training',
        'Higher = faster convergence, risk of instability',
        'More = faster training (up to CPU limit)',
        'Skip-gram better for rare words, CBOW faster'
    ]
}

hyperparams_df = pd.DataFrame(hyperparams_data)
print("📋 FastText Hyperparameters Reference:")
print("=" * 100)
for _, row in hyperparams_df.iterrows():
    print(f"🔧 {row['Parameter']} ({row['Default']})")
    print(f"   {row['Description']}")
    print(f"   Range: {row['Range/Options']}")
    print(f"   Impact: {row['Impact']}")
    print()

# Scaling recommendations
print("📈 Scaling Recommendations for Larger Corpora:")
print("""
1. **Small corpus (<1MB)**: Use default parameters, increase epochs to 10-20
2. **Medium corpus (1MB-100MB)**: Increase dim to 100-200, use 8-16 threads  
3. **Large corpus (>100MB)**: 
   - dim: 200-300
   - epoch: 5-10 (sufficient for convergence)
   - thread: Use all available CPU cores
   - Consider using hierarchical softmax for very large vocabularies
4. **Memory constraints**: Reduce dim, use smaller n-gram range (minn=3, maxn=4)
5. **Speed optimization**: Use CBOW model, reduce ws to 3-5, use more threads
""")

## Limitations & Caveats

Three of the results above are partly artifacts of the ten-sentence corpus, and it is worth
being explicit about which, because the same code on a real corpus behaves differently.

**1. The nearest neighbours are noise.** `get_nearest_neighbors` ranks against a vocabulary
of a few dozen words that each occurred once or twice. With that little co-occurrence
evidence the skip-gram objective has nothing to fit, so the neighbours you see are driven
almost entirely by shared character n-grams -- which is why `fasttext` lands near `fast`
and `preprocessing` near `processing`. That is a real demonstration of *subword* similarity
and it is not semantic similarity. On a Wikipedia-scale corpus the two start to agree; here
they cannot.

**2. The t-SNE plot should not be read as a semantic map.** The word list above has 17
entries, and the perplexity guard in `visualize_embeddings` settles on 16 -- one less than
the sample count. Perplexity is meant to be an effective neighbourhood size, so asking for
16 out of 17 means every point is a neighbour of every other and there is no local structure
left to preserve. Distances and cluster shapes in that figure are therefore not
interpretable. The plot is included because the code is the code you would run on real
embeddings; the picture it produces here is decoration.

**3. The threading benchmark measures startup, not throughput.** The corpus is a few hundred
bytes. Training on it takes milliseconds, so nearly all of the wall-clock time in that table
is process setup, memory allocation and thread spawning. That is why the "speedup" column
can come out at or below 1.0x: adding threads adds coordination cost with no work to spread
across them. Multi-threading in FastText is genuinely close to linear, but you need a corpus
of at least tens of megabytes before a benchmark can show it. Re-run that cell on a real
corpus before quoting any speedup number from it.

**4. Subword vectors are hashed, and collisions are silent.** Every n-gram is mapped into a
fixed number of buckets (`bucket`, default 2,000,000). Two unrelated n-grams that hash to
the same bucket share a vector, and nothing warns you. On a small corpus collisions are
rare; on a large multilingual one they are routine, and they are one reason a rare word's
embedding can be worse than its n-grams suggest.

**5. The results are reproducible, but not deterministic across machines.** The t-SNE call
is seeded with `random_state=42`, but FastText's own training uses multiple threads with
unsynchronised updates, so vector values differ slightly between runs even with the same
data. Set `thread=1` if you need bit-identical embeddings; expect it to be slower.

**6. Nothing here is evaluated.** There is no downstream task, no held-out set and no
comparison against Word2Vec or GloVe -- so the notebook shows *how* FastText represents
words, not *how well*. Any claim about FastText being better than an alternative has to come
from a benchmark, and the next-steps list below is where to start building one.

## Conclusion & Further Reading

### What We Learned

In this notebook, we explored FastText's subword approach to word embeddings:

1. **Subword Tokenization**: How character n-grams capture morphological structure
2. **Model Training**: Skip-gram vs. CBOW architectures and key hyperparameters  
3. **Subword Analysis**: How individual n-grams contribute to final word vectors
4. **Performance**: Multi-threading benefits for training speed
5. **Visualization**: t-SNE for understanding semantic relationships
6. **OOV Handling**: Robust embeddings for unseen words via subword composition

### Key Advantages of FastText

- ✅ **Morphological awareness**: Captures word formation patterns
- ✅ **OOV robustness**: Handles unseen words effectively  
- ✅ **Language flexibility**: Works well across different languages
- ✅ **Fast training**: Efficient implementation with multi-threading
- ✅ **Memory efficient**: Shared subword representations

### Next Steps

1. **Try supervised FastText**: Use `fasttext.train_supervised()` for classification tasks
2. **Compare architectures**: Benchmark FastText vs. Word2Vec vs. GloVe on your data
3. **Language-specific tuning**: Adjust n-gram ranges for different languages
4. **Downstream evaluation**: Test embeddings on specific NLP tasks
5. **Large-scale training**: Apply to Wikipedia or Common Crawl data

### Further Reading

- 📄 [Original FastText Paper](https://arxiv.org/abs/1607.04606) - Bojanowski et al. (2017)
- 🐙 [FastText GitHub Repository](https://github.com/facebookresearch/fastText)
- 📚 [FastText Documentation](https://fasttext.cc/docs/en/support.html)
- 🎓 [Subword Embeddings Survey](https://arxiv.org/abs/1901.08450) - Qiu et al. (2019)

Happy embedding! 🚀